# Modeling and Evaluation

In this section, we will build and evaluate our machine learning models. We will use the processed data from the previous steps and apply various modeling techniques to find the best-performing model for our task.


# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries & Data](#1.-Importing-Libraries-&-Data) <br><br>
    
2. [Exploratory Data Analysis](#2.-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br><br>
    
3. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

In [4]:
# ======================================================
# 🧠 30 — Model Training (Setup & Loads)
# ======================================================
import os
import json
import pandas as pd
import numpy as np

# Paths
data_dir = "../data/"
encoded_dir = os.path.join(data_dir, "encoded_data")
fs_dir = os.path.join(data_dir, "feature_selection")

# Load encoded & scaled splits (from Notebook 12)
X_train_scaled = pd.read_csv(os.path.join(encoded_dir, "12_X_train.csv"))
y_train = pd.read_csv(os.path.join(encoded_dir, "12_y_train.csv")).squeeze()

X_val_scaled   = pd.read_csv(os.path.join(encoded_dir, "12_X_val.csv"))
y_val = pd.read_csv(os.path.join(encoded_dir, "12_y_val.csv")).squeeze()

X_test_scaled  = pd.read_csv(os.path.join(encoded_dir, "12_X_test.csv"))

# Load selected features (from Notebook 20)
with open(os.path.join(fs_dir, "20_selected_features.json"), "r") as f:
    selected_features = json.load(f)

print("✅ Data & feature list loaded.")
print(f"X_train_scaled: {X_train_scaled.shape} | X_val_scaled: {X_val_scaled.shape} | X_test_scaled: {X_test_scaled.shape}")
print(f"#selected_features: {len(selected_features)}")


✅ Data & feature list loaded.
X_train_scaled: (56979, 20) | X_val_scaled: (18994, 20) | X_test_scaled: (32567, 20)
#selected_features: 10


In [5]:
# ======================================================
# Build final matrices from selected features
# ======================================================
# Ensure all selected features exist
missing_in_train = [c for c in selected_features if c not in X_train_scaled.columns]
if missing_in_train:
    print("⚠️ Missing features in X_train_scaled:", missing_in_train)

# Take the intersection as a safeguard (e.g., after later changes)
selected_features_final = [c for c in selected_features if c in X_train_scaled.columns]

X_train_final = X_train_scaled[selected_features_final].copy()
X_val_final   = X_val_scaled[selected_features_final].copy()
X_test_final  = X_test_scaled[selected_features_final].copy()

print("✅ Final matrices built.")
print(f"X_train_final: {X_train_final.shape} | X_val_final: {X_val_final.shape} | X_test_final: {X_test_final.shape}")

# Quick look
display(X_train_final.head(3))
display(y_train.head(3))


✅ Final matrices built.
X_train_final: (56979, 10) | X_val_final: (18994, 10) | X_test_final: (32567, 10)


,model,engineSize,year,mileage,Brand,mpg,transmission_manual,transmission_semi-auto,tax,transmission_automatic
0,-0.609686,-0.75,0.000000,0.105590,-0.137593,0.594406,0.0,0.0,0.25,0.0
1,0.097793,0.50,-0.333333,0.872986,0.266658,0.594406,-1.0,1.0,0.00,0.0
2,-0.202712,-0.75,0.666667,-0.334702,-0.137593,0.000000,0.0,0.0,0.00,0.0


0    10299.0
1    16494.0
2    15320.0
Name: price, dtype: float64

In [6]:
# ======================================================
# Utilities: metrics & evaluation
# ======================================================
from sklearn.metrics import mean_squared_error, r2_score

def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))  # compatible with older sklearn versions
    r2 = float(r2_score(y_true, y_pred))
    return {"RMSE": rmse, "R2": r2}

def report_model(name, y_true, y_pred):
    m = evaluate_regression(y_true, y_pred)
    print(f"[{name}]  RMSE: {m['RMSE']:,.2f} | R²: {m['R2']:.4f}")
    return m


In [7]:
# ======================================================
# Baseline Model 1 — Linear Regression
# ======================================================
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
lin.fit(X_train_final, y_train)

y_val_pred_lin = lin.predict(X_val_final)
metrics_lin = report_model("LinearRegression (baseline)", y_val, y_val_pred_lin)


[LinearRegression (baseline)]  RMSE: 4,620.94 | R²: 0.7683


In [8]:
# ======================================================
# Baseline Model 2 — RandomForest
# ======================================================
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_final, y_train)

y_val_pred_rf = rf.predict(X_val_final)
metrics_rf = report_model("RandomForest (baseline)", y_val, y_val_pred_rf)


[RandomForest (baseline)]  RMSE: 2,298.70 | R²: 0.9427


In [9]:
# ======================================================
# Compare baselines & pick current best
# ======================================================
import pandas as pd

cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "RandomForest",     **metrics_rf},
]).sort_values(by="RMSE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"🏆 Current best (validation): {best_name}")


,model,RMSE,R2
1,RandomForest,2298.701506,0.942670
0,LinearRegression,4620.943914,0.768326


🏆 Current best (validation): RandomForest


In [10]:
# ======================================================
# Compare baselines & pick current best
# ======================================================
import pandas as pd

cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "RandomForest",     **metrics_rf},
]).sort_values(by="RMSE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"🏆 Current best (validation): {best_name}")


,model,RMSE,R2
1,RandomForest,2298.701506,0.942670
0,LinearRegression,4620.943914,0.768326


🏆 Current best (validation): RandomForest


In [12]:
# ======================================================
# 🎯 Create Kaggle Submission (carID, price)
# ======================================================
import os
import pandas as pd
import numpy as np
from datetime import datetime

# 1) Load the raw test file to obtain carID
test_raw_path = os.path.join(data_dir, "test.csv")
test_raw = pd.read_csv(test_raw_path)

assert "carID" in test_raw.columns, "carID not found in ../data/test.csv"
assert len(test_raw) == len(X_test_final), "Length mismatch between test.csv and X_test_final!"

car_ids = test_raw["carID"].reset_index(drop=True)

# 2) Ensure a best_model exists (fallback if cell 7 was not executed)
try:
    best_model
except NameError:
    if "rf" in globals():
        best_model = rf
        print("ℹ️ Using RandomForest as best_model (fallback).")
    elif "lin" in globals():
        best_model = lin
        print("ℹ️ Using LinearRegression as best_model (fallback).")
    else:
        raise RuntimeError("No trained model found. Please run the training cells first.")

# 3) Generate predictions
y_test_pred = best_model.predict(X_test_final)

# (Optional) Round/clip according to competition rules
# Here we round to whole units, as in the sample, without allowing negative prices:
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)
y_test_pred_rounded = np.rint(y_test_pred).astype(int)

# 4) Build the submission DataFrame
submission = pd.DataFrame({
    "carID": car_ids,
    "price": y_test_pred_rounded  # optionally switch to y_test_pred if floats are allowed or preferred
})

# 5) Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}.csv")
submission.to_csv(sub_path, index=False)

print(f"✅ Submission saved to: {sub_path}")
display(submission.head(10))


✅ Submission saved to: ../data/submissions/30_submission_20251026_1713.csv


,carID,price
0,89856,20217
1,106581,26693
2,80886,13018
3,100174,19849
4,81376,23974
5,85391,13905
6,82175,14687
7,95250,15214
8,85071,4980
9,96210,18133
